In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "manishvem/signatures-dataset"
)

print(dataset_path)

100%|██████████| 1.38G/1.38G [00:41<00:00, 36.0MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/manishvem/signatures-dataset/versions/1


In [ ]:
#Inspect dataset structure
import os

dataset_path = "/root/.cache/kagglehub/datasets/manishvem/signatures-dataset/versions/1/signature_ds_combined"

folders = os.listdir(dataset_path)

print(len(folders))
print(folders[:10])

2974
['0414', '0575_forg', '0335', '1423_forg', '0799', '0522', '1298', '0655_forg', '1036_forg', '0639_forg']


In [ ]:
#Load genuine signatures for GANs training
import cv2
import numpy as np
import os

genuine_images = []

for folder in os.listdir(dataset_path):

    if "_forg" in folder:
        continue

    folder_path = os.path.join(dataset_path, folder)

    for file in os.listdir(folder_path):
        # Check if the file is an image (e.g., has a .png, .jpg, or .jpeg extension)
        if not (file.endswith('.png') or file.endswith('.jpg') or file.endswith('.jpeg')):
            continue # Skip non-image files

        img_path = os.path.join(folder_path, file)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Ensure image was loaded successfully before resizing
        if img is not None:
            img = cv2.resize(img, (64,64))

            img = img / 255.0

            genuine_images.append(img)


In [ ]:
#Convert to array
genuine_images = np.array(genuine_images)
genuine_images = genuine_images.reshape(-1,64,64,1)

print(genuine_images.shape)

(16607, 64, 64, 1)


Why only genuine signatures used here?

Because GAN learns:

distribution of real signatures

Later:

Generator produces realistic forged signatures

In [ ]:
#Building GANs Generator to create synthetic images
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, Conv2DTranspose

def build_generator():

    model = Sequential()

    model.add(Dense(8*8*128,input_dim=100))

    model.add(Reshape((8,8,128)))

    model.add(Conv2DTranspose(128,4,strides=2,padding="same",activation="relu"))

    model.add(Conv2DTranspose(64,4,strides=2,padding="same",activation="relu"))

    model.add(Conv2DTranspose(1,4,strides=2,padding="same",activation="sigmoid"))

    return model

In [ ]:
#Building GANs discriminators to detect genuine and fake images
from tensorflow.keras.layers import Conv2D, Flatten, Dense, LeakyReLU

def build_discriminator():

    model = Sequential()

    model.add(Conv2D(64,4,strides=2,input_shape=(64,64,1)))

    model.add(LeakyReLU())

    model.add(Conv2D(128,4,strides=2))

    model.add(LeakyReLU())

    model.add(Flatten())

    model.add(Dense(1,activation="sigmoid"))

    return model

In [ ]:
#Compile GANs
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

generator = build_generator()

discriminator = build_discriminator()

discriminator.compile(
    optimizer=Adam(0.0002),
    loss="binary_crossentropy"
)

discriminator.trainable = False

gan_input = Input(shape=(100,))

fake_signature = generator(gan_input)

gan_output = discriminator(fake_signature)

gan = Model(gan_input,gan_output)

gan.compile(
    optimizer=Adam(0.0002),
    loss="binary_crossentropy"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Training GANs
epochs = 2000
batch_size = 32

for epoch in range(epochs):

    idx = np.random.randint(0, genuine_images.shape[0], batch_size)

    real = genuine_images[idx]

    noise = np.random.normal(0, 1, (batch_size, 100))

    fake = generator.predict(noise, verbose=0)

    d_loss_real = discriminator.train_on_batch(real, np.ones((batch_size,1)))

    d_loss_fake = discriminator.train_on_batch(fake, np.zeros((batch_size,1)))

    noise = np.random.normal(0, 1, (batch_size, 100))

    g_loss = gan.train_on_batch(noise, np.ones((batch_size,1)))

    if epoch % 50 == 0:

        print(f"Epoch {epoch}")
        print(f"D_loss_real: {d_loss_real}")
        print(f"D_loss_fake: {d_loss_fake}")
        print(f"G_loss: {g_loss}")

/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Epoch 0
D_loss_real: 0.6566816568374634
D_loss_fake: 0.6845172643661499
G_loss: 0.6743838787078857
Epoch 50
D_loss_real: 0.6891329288482666
D_loss_fake: 0.6895712018013
G_loss: 0.6665019392967224
Epoch 100
D_loss_real: 0.6991035342216492
D_loss_fake: 0.6995505094528198
G_loss: 0.6483152508735657
Epoch 150
D_loss_real: 0.7151709198951721
D_loss_fake: 0.715774416923523
G_loss: 0.621676504611969
Epoch 200
D_loss_real: 0.7488340139389038
D_loss_fake: 0.7499951124191284
G_loss: 0.5761160254478455
Epoch 250
D_loss_real: 0.8061271905899048
D_loss_fake: 0.8076927661895752
G_loss: 0.5171941518783569
Epoch 300
D_loss_real: 0.8719242811203003
D_loss_fake: 0.8735812902450562
G_loss: 0.4632146954536438
Epoch 350
D_loss_real: 0.9345061182975769
D_loss_fake: 0.9360803961753845
G_loss: 0.4187641739845276
Epoch 400
D_loss_real: 0.9902366399765015
D_loss_fake: 0.9917027950286865
G_loss: 0.3827676773071289
Epoch 450
D_loss_real: 1.039027452468872
D_loss_fake: 1.040358066558838
G_loss: 0.3533232808113098


In [ ]:
#Generate synthetic signatures
noise = np.random.normal(0,1,(20,100))

generated_images = generator.predict(noise)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step


In [ ]:
#Save so that synthetic dataset is ready
for i,img in enumerate(generated_images):

    cv2.imwrite(
        f"generated_signature_{i}.png",
        img*255
    )

In [ ]:
#Load all signatures real+forged+synthetic
import os
import cv2
import numpy as np

dataset_path = "/root/.cache/kagglehub/datasets/manishvem/signatures-dataset/versions/1/signature_ds_combined"

signature_data = {}

for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    images = []

    for file in os.listdir(folder_path):
        # Check if the file is an image (e.g., has a .png, .jpg, or .jpeg extension)
        if not (file.endswith('.png') or file.endswith('.jpg') or file.endswith('.jpeg')):
            continue # Skip non-image files

        img_path = os.path.join(folder_path, file)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Ensure image was loaded successfully before resizing
        if img is not None:
            img = cv2.resize(img, (64,64))

            img = img / 255.0

            images.append(img)

    signature_data[folder] = images


In [ ]:
#Add GAN generated images in the dataset
generated_folder = "/content"

generated_images = []

for file in os.listdir(generated_folder):

    if "generated_signature" in file:

        img = cv2.imread(os.path.join(generated_folder,file),0)

        img = cv2.resize(img,(64,64))

        img = img/255.0

        generated_images.append(img)

In [ ]:
signature_data["synthetic_forg"] = generated_images

In [ ]:
#Creating Siamese training pairs
pairs = []
labels = []

writers = list(signature_data.keys())

for writer in writers:

    if "_forg" in writer or writer == "synthetic_forg":
        continue

    genuine = signature_data[writer]

    forged = signature_data.get(writer+"_forg", [])

    synthetic = signature_data.get("synthetic_forg", [])

    # genuine vs genuine (positive)

    for i in range(len(genuine)-1):

        pairs.append([genuine[i], genuine[i+1]])

        labels.append(1)

    # genuine vs forged (negative)

    for i in range(min(len(genuine),len(forged))):

        pairs.append([genuine[i], forged[i]])

        labels.append(0)

    # genuine vs synthetic forged (negative)

    for i in range(min(len(genuine),len(synthetic))):

        pairs.append([genuine[i], synthetic[i]])

        labels.append(0)

In [ ]:
pairs = np.array(pairs)
labels = np.array(labels)

In [ ]:
#Train test split
from sklearn.model_selection import train_test_split

x1 = pairs[:,0].reshape(-1,64,64,1)
x2 = pairs[:,1].reshape(-1,64,64,1)

x1_train,x1_test,x2_train,x2_test,y_train,y_test = train_test_split(
    x1,
    x2,
    labels,
    test_size=0.2
)

In [ ]:
#Build Siamese encoder network
from tensorflow.keras.layers import Input,Conv2D,MaxPooling2D,Flatten,Dense
from tensorflow.keras.models import Model

def build_encoder():

    input = Input((64,64,1))

    x = Conv2D(32,(3,3),activation='relu')(input)

    x = MaxPooling2D()(x)

    x = Conv2D(64,(3,3),activation='relu')(x)

    x = MaxPooling2D()(x)

    x = Conv2D(128,(3,3),activation='relu')(x)

    x = MaxPooling2D()(x)

    x = Flatten()(x)

    x = Dense(128,activation='relu')(x)

    return Model(input,x)

In [ ]:
#Building Siamese similarity model
import tensorflow as tf
from tensorflow.keras.layers import Lambda

def euclidean_distance(vectors):

    x,y = vectors

    return tf.sqrt(tf.reduce_sum(tf.square(x-y),axis=1,keepdims=True))

In [ ]:
#Final architecture
encoder = build_encoder()

input_a = Input((64,64,1))

input_b = Input((64,64,1))

embedding_a = encoder(input_a)

embedding_b = encoder(input_b)

distance = Lambda(euclidean_distance)([embedding_a,embedding_b])

output = Dense(1,activation="sigmoid")(distance)

siamese_model = Model([input_a,input_b],output)

siamese_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [ ]:
#Train Siamese Model
siamese_model.fit(
    [x1_train,x2_train],
    y_train,
    validation_data=([x1_test,x2_test],y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 325s 285ms/step - accuracy: 0.6681 - loss: nan - val_accuracy: 0.6590 - val_loss: nan
Epoch 2/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 321s 284ms/step - accuracy: 0.6608 - loss: nan - val_accuracy: 0.6590 - val_loss: nan
Epoch 3/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 320s 283ms/step - accuracy: 0.6608 - loss: nan - val_accuracy: 0.6590 - val_loss: nan
Epoch 4/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 323s 283ms/step - accuracy: 0.6608 - loss: nan - val_accuracy: 0.6590 - val_loss: nan
Epoch 5/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 320s 283ms/step - accuracy: 0.6608 - loss: nan - val_accuracy: 0.6590 - val_loss: nan
Epoch 6/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 321s 284ms/step - accuracy: 0.6608 - loss: nan - val_accuracy: 0.6590 - val_loss: nan
Epoch 7/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 351s 309ms/step - accuracy: 0.6608 - loss: nan - val_accuracy: 0.6590 - val_loss: nan
Epoch 8/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 354s 313ms/step - accuracy: 0.6608 - loss: nan - val_accuracy:

In [ ]:
siamese_model.evaluate([x1_test,x2_test],y_test)

283/283 ━━━━━━━━━━━━━━━━━━━━ 25s 88ms/step - accuracy: 0.6590 - loss: nan


[nan, 0.6589729189872742]

In [ ]:
prediction = siamese_model.predict([x1_test[0:1],x2_test[0:1]])

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step
[[nan]]


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Lambda
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.models import Model

In [ ]:
#Building Encoder network
def build_encoder():

    input_layer = Input((64,64,1))

    x = Conv2D(32,(3,3),activation='relu')(input_layer)
    x = MaxPooling2D()(x)

    x = Conv2D(64,(3,3),activation='relu')(x)
    x = MaxPooling2D()(x)

    x = Conv2D(128,(3,3),activation='relu')(x)
    x = MaxPooling2D()(x)

    x = Flatten()(x)

    x = Dense(128,activation='relu')(x)

    x = BatchNormalization()(x)

    return Model(input_layer,x)

In [ ]:
#Safe Euclidean distance function (fixes NaN loss)
def euclidean_distance(vectors):

    x, y = vectors

    sum_square = tf.reduce_sum(
        tf.square(x - y),
        axis=1,
        keepdims=True
    )

    return tf.sqrt(tf.maximum(sum_square, 1e-10))

In [ ]:
#Build siamese network
encoder = build_encoder()

input_a = Input((64,64,1))
input_b = Input((64,64,1))

embedding_a = encoder(input_a)
embedding_b = encoder(input_b)

distance = Lambda(euclidean_distance)([embedding_a, embedding_b])

output = Dense(1, activation="sigmoid")(distance)

siamese_model = Model([input_a,input_b], output)

In [ ]:
#Compliling model with safe optmizer means
siamese_model.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    metrics=["accuracy"]
)

In [ ]:
#Verifying dataset has no NaN values
import numpy as np

print(np.isnan(x1_train).sum())
print(np.isnan(x2_train).sum())

0
0


In [ ]:
#Training siamese model
history = siamese_model.fit(

    [x1_train, x2_train],

    y_train,

    validation_data=([x1_test, x2_test], y_test),

    epochs=10,

    batch_size=32
)

Epoch 1/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 350s 306ms/step - accuracy: 0.6608 - loss: 0.5578 - val_accuracy: 0.6588 - val_loss: 0.4980
Epoch 2/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 354s 313ms/step - accuracy: 0.6612 - loss: 0.5197 - val_accuracy: 0.6673 - val_loss: 0.4824
Epoch 3/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 374s 306ms/step - accuracy: 0.6656 - loss: 0.5071 - val_accuracy: 0.7031 - val_loss: 0.4610
Epoch 4/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 375s 299ms/step - accuracy: 0.6755 - loss: 0.4992 - val_accuracy: 0.7372 - val_loss: 0.4593
Epoch 5/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 400s 315ms/step - accuracy: 0.6879 - loss: 0.4900 - val_accuracy: 0.7505 - val_loss: 0.4508
Epoch 6/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 377s 333ms/step - accuracy: 0.6995 - loss: 0.4864 - val_accuracy: 0.7724 - val_loss: 0.4448
Epoch 7/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 354s 313ms/step - accuracy: 0.7079 - loss: 0.4811 - val_accuracy: 0.7747 - val_loss: 0.4384
Epoch 8/10
1132/1132 ━━━━━━━━━━━━━━━━━━━━ 413s 340ms/step - ac

In [ ]:
siamese_model.evaluate([x1_test, x2_test], y_test)

283/283 ━━━━━━━━━━━━━━━━━━━━ 22s 78ms/step - accuracy: 0.7877 - loss: 0.4217


[0.4217446744441986, 0.7877416014671326]

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

predictions = siamese_model.predict([x1_test, x2_test])
final_predictions = np.where(predictions > 0.5, 1, 0)

cm = confusion_matrix(y_test, final_predictions)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, final_predictions))

283/283 ━━━━━━━━━━━━━━━━━━━━ 22s 78ms/step
Confusion Matrix:
[[4970  997]
 [ 925 2163]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.83      0.84      5967
           1       0.68      0.70      0.69      3088

    accuracy                           0.79      9055
   macro avg       0.76      0.77      0.77      9055
weighted avg       0.79      0.79      0.79      9055

